# Voice and live transcript

This notebook explains how the mock interviewer **speaks**, how the candidate
**answers by voice or text**, and how the **live transcript** stays synchronized.

**Source files**
- `app.py` — widget layout and event wiring (`iv_audio`, `iv_answer_audio`, `iv_chatbot`)
- `careercrew/callbacks_interview.py` — `interview_start`, `_ask_question`, `interview_submit`, `_turns_to_messages`
- `careercrew/audio.py` — `speak`, `transcribe`


## How it works

When the interviewer asks a question, CareerCrew uses the **same text string**
twice: TTS synthesizes speech for listening, and `append_turn` adds that text
to the transcript for reading.

When the candidate answers, Gradio saves the microphone recording to a temporary
filepath. Whisper converts it to text (or the candidate can type). **Only that
text** is sent to the evaluation LLM. The answer is appended to the transcript,
then the next question is spoken and displayed the same way.

Voice and transcript are always the same underlying text — audio is I/O around it.


## Architecture diagram

```
AGENT SPEAKS                          CANDIDATE ANSWERS
───────────                           ─────────────────
question.text                         mic → Gradio temp filepath
     │                                      │
     ├─► audio.speak() → .mp3               ├─► audio.transcribe() → TEXT
     │   → gr.Audio(autoplay)               │         OR typed Textbox
     │                                      │
     └─► append_turn(INTERVIEWER)           └─► append_turn(CANDIDATE)
              │                                      │
              └────────── Chatbot ←──────────────────┘
                   (_turns_to_messages)

Chat LLM / evaluate_answer sees TEXT ONLY — never the audio bytes.
```


## 1. Gradio UI widgets

Defined in `app.py` (Mock interview tab widgets, wired to `callbacks_interview`):

| Widget | Role |
|--------|------|
| `iv_audio` | Interviewer player (`autoplay=True`, not interactive) |
| `iv_answer_audio` | Candidate mic (`sources=["microphone"]`, `type="filepath"`) |
| `iv_answer_text` | Optional typed fallback |
| `iv_chatbot` | Live transcript |


In [ ]:
# Widget configuration used in app.py (illustrative)

ui = {
    "iv_audio": {
        "label": "Interviewer (listen)",
        "autoplay": True,
        "interactive": False,  # plays TTS mp3 from _ask_question
    },
    "iv_answer_audio": {
        "sources": ["microphone"],
        "type": "filepath",  # Gradio passes a temp path into interview_submit
        "label": "Record your answer",
    },
    "iv_answer_text": {
        "label": "Or type your answer",
        "optional": True,
    },
    "iv_chatbot": {
        "label": "Interview transcript",
        "fed_by": "_turns_to_messages(session)",
    },
}

for name, cfg in ui.items():
    print(f"{name:16} -> {cfg}")


## 2. Agent speaks = TTS + transcript turn

`_ask_question` in `careercrew/callbacks_interview.py`:

1. `audio.speak(question.text)` → temporary `.mp3` (or `None` if no API key)
2. `interview.append_turn(..., INTERVIEWER, question.text)`
3. Return the audio path to `iv_audio` and `_turns_to_messages(session)` to `iv_chatbot`

`audio.speak` (`careercrew/audio.py`) calls OpenAI **`tts-1`**, writes a temp mp3,
and returns the path. On failure it returns `None` and the UI stays text-only.


In [ ]:
# Simulate _ask_question transcript behavior (no API calls)

def speak_stub(text: str):
    """Stand-in for audio.speak — real code writes a temp .mp3."""
    print(f"[TTS] would synthesize: {text!r}")
    return "/tmp/careercrew_q_demo.mp3"

turns = []

def append_turn(role: str, text: str):
    turns.append({"role": role, "text": text})

question_text = "Tell me about a time you owned an API end-to-end."
audio_path = speak_stub(question_text)
append_turn("interviewer", question_text)

print("audio_path ->", audio_path)
print("turns      ->", turns)


## 3. Candidate answers = mic filepath → Whisper → text

`interview_submit` in `careercrew/callbacks_interview.py`:

```python
transcript = audio.transcribe(answer_audio) if answer_audio else ""
answer = transcript.strip() or (answer_text or "").strip()
```

- Gradio recorded the mic and passed **`answer_audio`** as a filepath.
- `audio.transcribe` opens the file and calls OpenAI **`whisper-1`**.
- Voice transcript is preferred; if empty, typed text is used.
- Then `validate_interview_answer` and `evaluate_answer` run on **text only**.


In [ ]:
# Voice transcript preferred over typed text when both exist

def transcribe_stub(path):
    """Stand-in for audio.transcribe — real code calls whisper-1."""
    if not path:
        return ""
    return "I built a FastAPI service, owned deploy, and cut latency 40%."

answer_audio = "/tmp/gradio_mic_xyz.wav"   # Gradio temp filepath
answer_text = "short typed draft"         # optional

transcript = transcribe_stub(answer_audio) if answer_audio else ""
answer = transcript.strip() or (answer_text or "").strip()

print("final answer to LLM ->", answer)
print("note: chat model never sees", answer_audio)


## 4. Live transcript sync

`_turns_to_messages` (`careercrew/callbacks_interview.py`) maps session turns to Gradio Chatbot messages:

| Session role | Chatbot role |
|--------------|--------------|
| `INTERVIEWER` | `"assistant"` |
| `CANDIDATE` | `"user"` |

The Chatbot is refreshed after every ask and every submit so it stays
synchronized with the spoken dialogue.


In [ ]:
# Same mapping as callbacks_interview._turns_to_messages

def turns_to_messages(turns):
    messages = []
    for turn in turns:
        role = "assistant" if turn["role"] == "interviewer" else "user"
        messages.append({"role": role, "content": turn["text"]})
    return messages

append_turn("candidate", answer)
append_turn("interviewer", "How did you measure that latency improvement?")

chatbot_messages = turns_to_messages(turns)
print("What the Gradio Chatbot receives:")
print()
for m in chatbot_messages:
    who = "Interviewer" if m["role"] == "assistant" else "You"
    print(f"  {who}: {m['content']}")


## 5. Failure-tolerant design

| Failure | Behavior |
|---------|----------|
| No OpenAI key | `audio_available()` is false → text-only interview |
| TTS fails | Question still shown as HTML text |
| STT fails / no mic | Typed `iv_answer_text` still works |
| Empty answer | Guardrail rejects; candidate is asked again |

`audio.py` catches exceptions and returns `""` / `None` so audio problems
never crash the UI.


## Code reference

| Step | File | Symbol |
|------|------|--------|
| Mic + speaker UI | `app.py` | `iv_answer_audio`, `iv_audio`, `iv_chatbot` |
| Ask (TTS + turn) | `callbacks_interview.py` | `_ask_question` |
| Submit (STT + eval) | `callbacks_interview.py` | `interview_submit` |
| Transcript map | `callbacks_interview.py` | `_turns_to_messages` |
| Whisper / TTS | `careercrew/audio.py` | `transcribe`, `speak` |
| Persist text | `session_store.py` | `save_interview` |

Continue with `02_mock_interview_flow.ipynb` for the full interview loop.
